# SenGuard-MSC · Notebook 07 — Virtual passage: predicting a culture's later expression

Can an early sample predict the same culture's expression profile at a later passage? This is the predictive part of the twin. The rules below are written and hashed before any prediction is evaluated.

In [ ]:
AMENDMENT2 = """# Analysis protocol amendment 2: virtual passage

Fixed on 2026-11-01, before any prediction below was evaluated. It follows a failed per-donor forecast (deliverables/tables/forecast_backtest.json), so the idea of learning across cohorts was chosen after seeing that failure.

## Question
From one early sample of a culture, predict its within-sample gene ranks at a later passage.

## Models
- Carry-forward: no change. This is the baseline to beat.
- Population drift: a per-gene drift rate learned from other cohorts, multiplied by the change in log2(passage + 1).
- Personal drift: population drift scaled by how fast this donor moved between its first two passages.

## Validation
Leave one cohort out. Cohorts: GSE309666, GSE178514, GSE137186, GSE275980, GSE194298, GSE233634, GSE119987, GSE56362. GSE125632 is excluded because it failed the identity check. Unit: donor (medium arms averaged). Earliest available passage predicts last available passage.

## Gate for the interface
Population drift enters the interface only if all of these hold:
1. Gene-level RMSE beats carry-forward in at least 70% of held-out donors.
2. It wins in at least 6 of the 8 cohorts, by median error.
3. Predicted SenCat rank score has lower absolute error than carry-forward in at least 70% of held-out donors.
4. Median correlation between predicted and actual change across genes is at least 0.3.
5. Negative control: drift learned from passages shuffled within donors beats carry-forward in at most 60% of donors on RMSE, on average.

Personal drift is used only if it beats population drift in at least 60% of donors with 3 or more passages, on the later step.

## What this does not claim
It predicts transcript ranks. It does not predict potency, growth arrest, safety or cell behaviour. Remaining expansion capacity is examined only descriptively, because too few lineages have a growth endpoint.
"""

In [ ]:
HELD = ['GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362']
(REPORTS / 'ANALYSIS_AMENDMENT_2.md').write_text(AMENDMENT2, encoding='utf-8')
print('Amendment 2 sha256', sg.sha256(REPORTS / 'ANALYSIS_AMENDMENT_2.md'))
missing = [k for k in HELD if k not in COHORTS]
print('Missing cohorts:', missing)
HELD = [k for k in HELD if k in COHORTS]
genes = None
for k in HELD:
    cols = set(COHORTS[k]['sym'].columns)
    genes = cols if genes is None else genes & cols
genes = sorted(genes)
print('Shared genes', len(genes))
PROFILES = {}
for k in HELD:
    c = COHORTS[k]
    R = c['sym'][genes].rank(axis=1, pct=True)
    d = R.join(c['meta'][['subject', 'unit', 'passage']]).groupby(['subject', 'unit', 'passage'], as_index=False).mean(numeric_only=True)
    subs = []
    for (s, u), g in d.groupby(['subject', 'unit']):
        g = g.sort_values('passage')
        if g.passage.nunique() >= 2:
            subs.append(dict(subject=s, unit=u, passage=g.passage.to_numpy(float), X=g[genes].to_numpy(np.float32)))
    PROFILES[k] = subs
    print(k, 'subjects', len(subs))
tlog = lambda p: np.log2(np.asarray(p, float) + 1)
weights = pub_sym.reindex(genes).dropna()
w_idx = [genes.index(g) for g in weights.index]
w_rank = stats.rankdata(weights.to_numpy())
print('SenCat genes among shared', len(w_idx))
def rank_score(x):
    r = stats.rankdata(x[w_idx])
    return float(np.corrcoef(r, w_rank)[0, 1])

def learn_beta(cohorts, rng=None):
    per = []
    for k in cohorts:
        bs = []
        for s in PROFILES[k]:
            t = tlog(s['passage'])
            if rng is not None:
                t = rng.permutation(t)
            tc = t - t.mean()
            denom = (tc ** 2).sum()
            if denom > 0:
                bs.append((tc[None, :] @ (s['X'] - s['X'].mean(axis=0))).ravel() / denom)
        if bs:
            per.append(np.mean(bs, axis=0))
    return np.mean(per, axis=0)

rows, personal = [], []
betas = {}
for k in HELD:
    train = [c for c in HELD if c != k]
    beta = learn_beta(train)
    betas[k] = beta
    for s in PROFILES[k]:
        x0, xL = s['X'][0], s['X'][-1]
        dt = tlog(s['passage'][-1]) - tlog(s['passage'][0])
        y = xL - x0
        pred = beta * dt
        rmse_m, rmse_0 = float(np.sqrt(np.mean((y - pred) ** 2))), float(np.sqrt(np.mean(y ** 2)))
        sc_true, sc_0, sc_m = rank_score(xL), rank_score(x0), rank_score(x0 + pred)
        rows.append({'cohort': k, 'subject': s['subject'], 'unit': s['unit'], 'from': s['passage'][0], 'to': s['passage'][-1], 'dt': dt, 'rmse_model': rmse_m, 'rmse_carry': rmse_0, 'corr_change': float(np.corrcoef(pred, y)[0, 1]),
                     'score_true': sc_true, 'score_carry': sc_0, 'score_model': sc_m, 'err_score_model': abs(sc_m - sc_true), 'err_score_carry': abs(sc_0 - sc_true)})
        if len(s['passage']) >= 3:
            xm = s['X'][1]
            t0, tm, tL = tlog(s['passage'][0]), tlog(s['passage'][1]), tlog(s['passage'][-1])
            gamma = float(np.clip(((xm - x0) @ beta) / ((beta @ beta) * (tm - t0)), 0, 5))
            late = float(((xL - xm) @ beta) / ((beta @ beta) * (tL - tm)))
            y2 = xL - xm
            personal.append({'cohort': k, 'subject': s['subject'], 'unit': s['unit'], 'gamma_early': gamma, 'gamma_late': late, 'rmse_carry': float(np.sqrt(np.mean(y2 ** 2))), 'rmse_population': float(np.sqrt(np.mean((y2 - beta * (tL - tm)) ** 2))), 'rmse_personal': float(np.sqrt(np.mean((y2 - gamma * beta * (tL - tm)) ** 2)))})
res = pd.DataFrame(rows)
units = res.groupby(['cohort', 'unit'], as_index=False).mean(numeric_only=True)
units['model_better'] = units.rmse_model < units.rmse_carry
units['score_better'] = units.err_score_model < units.err_score_carry
units.to_csv(RESULTS / 'virtual_passage_units.csv', index=False)
print(units[['cohort', 'unit', 'from', 'to', 'rmse_carry', 'rmse_model', 'corr_change', 'err_score_carry', 'err_score_model', 'model_better', 'score_better']].round(3).to_string(index=False))
by = units.groupby('cohort').agg(donors=('unit', 'size'), rmse_carry=('rmse_carry', 'median'), rmse_model=('rmse_model', 'median'), corr=('corr_change', 'median'), better=('model_better', 'sum'), score_better=('score_better', 'sum'), err_carry=('err_score_carry', 'median'), err_model=('err_score_model', 'median'))
print(by.round(3).to_string())

rng = np.random.default_rng(sg.SEED)
neg = []
for rep in range(20):
    wins = []
    for k in HELD:
        beta = learn_beta([c for c in HELD if c != k], rng=rng)
        for s in PROFILES[k]:
            y = s['X'][-1] - s['X'][0]
            dt = tlog(s['passage'][-1]) - tlog(s['passage'][0])
            wins.append(np.mean((y - beta * dt) ** 2) < np.mean(y ** 2))
    neg.append(float(np.mean(wins)))
pers = pd.DataFrame(personal)
pers_u = pers.groupby(['cohort', 'unit'], as_index=False).mean(numeric_only=True)
pers_u.to_csv(RESULTS / 'virtual_passage_personal.csv', index=False)
print(pers_u.round(4).to_string(index=False))
persist = float(stats.spearmanr(pers_u.gamma_early, pers_u.gamma_late).statistic) if len(pers_u) > 2 else float('nan')

gate = {
    'donors': len(units),
    'share_rmse_better': float(units.model_better.mean()),
    'cohorts_better': int((by.rmse_model < by.rmse_carry).sum()), 'cohorts': len(by),
    'share_score_better': float(units.score_better.mean()),
    'median_corr_change': float(units.corr_change.median()),
    'negative_control_share': float(np.mean(neg)),
    'median_rmse_reduction': float(1 - (units.rmse_model / units.rmse_carry).median()),
    'median_err_score_model': float(units.err_score_model.median()), 'median_err_score_carry': float(units.err_score_carry.median()),
    'personal_share_better': float((pers_u.rmse_personal < pers_u.rmse_population).mean()), 'personal_donors': len(pers_u), 'rate_persistence_spearman': persist,
    'population_better_than_carry_later_step': float((pers_u.rmse_population < pers_u.rmse_carry).mean()),
}
gate['passes'] = bool(gate['share_rmse_better'] >= 0.7 and gate['cohorts_better'] >= 6 and gate['share_score_better'] >= 0.7 and gate['median_corr_change'] >= 0.3 and gate['negative_control_share'] <= 0.6)
gate['personal_passes'] = bool(gate['personal_share_better'] >= 0.6)
print(json.dumps(gate, indent=1))

# Error for the interface, by how far ahead the prediction reaches
units['dt_bin'] = pd.cut(units.dt, [0, 1, 2, 10], labels=['under 1', '1 to 2', 'over 2'])
err = units.groupby('dt_bin', observed=True).agg(n=('unit', 'size'), rmse=('rmse_model', 'median'), score_err=('err_score_model', 'median'), score_err_p90=('err_score_model', lambda v: float(np.percentile(v, 90))))
print(err.round(4).to_string())
final_beta = learn_beta(HELD)
top = pd.Series(final_beta, index=genes).sort_values()
print('Falling with passage:', top.head(25).index.tolist())
print('Rising with passage:', top.tail(25).index[::-1].tolist())
e2fg2m = set(hall['HALLMARK_E2F_TARGETS']) | set(hall['HALLMARK_G2M_CHECKPOINT'])
print('Share of top 100 falling genes in E2F/G2M:', round(float(top.head(100).index.isin(e2fg2m).mean()), 3), '| rising:', round(float(top.tail(100).index.isin(e2fg2m).mean()), 3))
sym_to_ens = {}
gs = set(genes)
for e, sy in ens2sym_tbl.items():
    if sy in gs:
        sym_to_ens.setdefault(sy, []).append(e)
json.dump({'genes': genes, 'ensembl_by_symbol': sym_to_ens, 'beta': {'final': [float(f'{x:.4g}') for x in final_beta], **{f'without_{k}': [float(f'{x:.4g}') for x in betas[k]] for k in [ASC, BM] if k in betas}},
           'signature': {g: float(pub_sym[g]) for g in weights.index}, 'error': err.reset_index().astype({'dt_bin': str}).to_dict('records'), 'gate': gate}, open(RESULTS / 'virtual_passage_model.json', 'w'))
json.dump(gate, open(RESULTS / 'virtual_passage_gate.json', 'w'), indent=1)
(REPORTS / 'VIRTUAL_PASSAGE_RESULTS.md').write_text('# Virtual passage results\n\nGate: ' + json.dumps(gate, indent=1) + '\n\nBy cohort:\n' + by.round(3).to_string() + '\n\nBy distance:\n' + err.round(4).to_string() + '\n', encoding='utf-8')

# Descriptive only: does early movement relate to reaching growth arrest in the adipose cohort?
growth = {'A': 'arrested', 'B': 'arrested', 'C': 'arrested', 'D': 'stopped at p10', 'E': 'stopped at p10'}
g = [{'donor': s['subject'], 'status': growth.get(s['subject']), 'last_passage': s['passage'][-1], 'gamma_early': float(((s['X'][1] - s['X'][0]) @ betas[ASC]) / ((betas[ASC] @ betas[ASC]) * (tlog(s['passage'][1]) - tlog(s['passage'][0]))))} for s in PROFILES[ASC]]
print(pd.DataFrame(g).round(3).to_string(index=False))
pd.DataFrame(g).to_csv(RESULTS / 'virtual_passage_growth_descriptive.csv', index=False)